# Gaming Toxicity Detection

**Authors:** Beibarys Nyussupov, Ruby Ngo, Paola Calle, Jett Forward

In [1]:
# libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
import re 
import sys
sys.path.insert(0, str(Path("../..").resolve()))

# custom functions - tokenizer, step evaluator
from src.tokenizer import tokenize
from src.prep_evaluation_multi import eval_step

# reproducibility
seed = 7524
np.random.seed(seed)

# project root (notebooks/gaming/ - notebooks/ - project root)
PROJECT_ROOT = Path().resolve().parent.parent

In [2]:
# data directories
DATA_DIR_WOT  = PROJECT_ROOT / "data/raw_data/wot/"
DATA_DIR_DOTA = PROJECT_ROOT / "data/raw_data/dota/"

## World of Tanks

**World of Tanks**
| Class | Terminology |
|---|---|
| 0 | Non-Toxic |
| 1 | Insults and Flaming |
| 2 | Other Offensive Texts |
| 3 | Hate and Harassment |
| 4 | Threats |
| 5 | Extremism |

#### Inspect each split

In [3]:
# import data 
train = pd.read_csv(DATA_DIR_WOT / "train.csv")
val = pd.read_csv(DATA_DIR_WOT / "val.csv")

# test dataset 
test_text = pd.read_csv(DATA_DIR_WOT /  "test_index_text.csv")
test_label = pd.read_csv(DATA_DIR_WOT / "test_index_label.csv")

# check the data 
for name, df in [("train", train), ("val", val), ("test_text", test_text), ("test_label", test_label)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"First few rows of the dataset:\n{df.head(3)}")
    print()

--- train ---
Shape: (42959, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

--- val ---
Shape: (5367, 3)
Columns: ['index', 'message', 'label']
First few rows of the dataset:
   index       message  label
0  31697  e50 is there    0.0
1  52311   this scouts    0.0
2   2775       i guess    0.0

--- test_text ---
Shape: (5375, 2)
Columns: ['index', 'message']
First few rows of the dataset:
   index         message
0  17775  aim you monkey
1  28228   Pz kill artys
2  19888             ggs

--- test_label ---
Shape: (5375, 2)
Columns: ['index', 'label']
First few rows of the dataset:
   index  label
0  17775    2.0
1  28228    0.0
2  19888    0.0



The test set is stored in two separate files - text and labels indexed by the same `index` column. Train and val are already complete. We need to join the test files on `index` before merging everything.

#### Reconstruct test split and merge all

In [4]:
# merge test text and labels
test = test_text.merge(test_label, on="index", how="inner")

# concatenate all data
wot = pd.concat([train, val, test], ignore_index=True)

print(f"Test join shape: {test.shape}\n")
print(f"Merged shape: {wot.shape}\n")
print(f"First few rows of the data:\n{wot.head(3)}\n")
print(f"Information about the data: {wot.info()}")

Test join shape: (5375, 3)

Merged shape: (53701, 3)

First few rows of the data:
   index                        message  label
0  30702                        no rush    0.0
1  18607  whatever ... watch the replay    0.0
2  32901                        useless    1.0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53701 entries, 0 to 53700
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   index    53701 non-null  int64  
 1   message  53701 non-null  object 
 2   label    53701 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.2+ MB
Information about the data: None


The test join uses `inner` - any index present in only one file would be silently dropped. If test shape after join is smaller than either file's length, there is an alignment issue in the original data. 

In [5]:
# to test improvement or degradation after each preprocessing step, 
# we export the data as raw parquet and compare score of predictions of raw vs processed data. 
wot.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

# baseline — before any cleaning
preprocessing_impact = eval_step("baseline_raw", datasets = ("WoT",))

        step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
baseline_raw     WoT 53701    0.4765       0.8608         0.553           0.4586             0.0           0.0              0.0


#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [6]:
# detecting Non-lating language first

# covers all major non-Latin scripts
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{wot[wot['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

# changes 
before = len(wot)
wot_en = wot[~wot["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(wot_en)} ({(before - len(wot_en))/before:.1%})")
print(f"Rows after the drop: {wot_en.shape[0]}")
# save new step again
wot_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)


Non-english documents:
      index                message  label
0     13087          ще злийте )))    1.0
1     31917       перший бій, сток    0.0
2     40395                ФОКУСЯТ    0.0
3     54412  елка спасибо тебе ...    0.0
4      5457                 уроди!    0.0
...     ...                    ...    ...
3845   5453                 я кину    0.0
3846   5456                светите    0.0
3847  13081       в чём притензия?    0.0
3848   7331      вот и подкруточки    0.0
3849  50128  тут танки стоят уебак    0.0

[3850 rows x 3 columns]

Dropped 3850 (7.2%)
Rows after the drop: 49851


In [7]:
# check non-latin language drop
preprocessing_impact = eval_step("after_nonlatin_drop", preprocessing_impact, datasets="WoT")
preprocessing_impact

               step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_nonlatin_drop     WoT 49851    0.4946       0.8679        0.5919           0.4756          0.0181        0.0389            0.017


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.017


In [8]:
from lingua import Language, LanguageDetectorBuilder

# detector to find latin but non-english documents 
detector = (LanguageDetectorBuilder 
    # detect all other languages using latin
    .from_all_languages_with_latin_script()
    # if top-2 languages are within 25% confidence of each other - do not drop 
    .with_minimum_relative_distance(0.25)
    .build())

# function to detect such documents 
def is_latin_non_english(text):
    if len(str(text).strip()) < 30:
        return False  # too short - keep
    lang = detector.detect_language_of(str(text))
    return lang is not None and lang != Language.ENGLISH

# such documents 
mask = wot_en["message"].apply(is_latin_non_english)
with pd.option_context("display.max_rows", 800):
    print(f"Non-english documents:\n{wot_en[mask]}\n")
    print(f"How many documents with non-english text:{wot_en[mask].shape[0]}")

Non-english documents:
       index                                            message  label
364    14038                    Menj már nemm kapsz spotot.....    0.0
494    29761          no ayudaste a nadie shptk ahora no llores    1.0
573     7353                  3 dingi od pierdolonych papier??w    2.0
683    35518                Etwas Hilfe bei D4 wäre schon nett!    0.0
778     6397        Hilfe! Ich werde gleich auf A3 verpr??gelt!    0.0
907     6395        Hilfe! Ich werde gleich auf C1 verpr??gelt!    0.0
957     1934  jeszcze t54 zespotowany byl wczesniej wiec wid...    0.0
1109    7645                     how many dmg arty? 2k? XDDDDDD    0.0
1110   51836                   LOSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS    0.0
1184    1984                   5x ep... mehr muss man net sagen    0.0
1373    6449  Hali! A MAATT kl??n akt??v j??t??kosokat keres...    0.0
1425   30904                  ooo polaczek, no jakby inaczej XD    0.0
1527    5943  tu ecris ce que tu veux, dans la mesure 

Only 294 rows. Let's drop that as well. 


In [9]:
# dataset update
wot_en = wot_en[~mask].reset_index(drop = True)
# length after the drop 
print(f"Length after the drop:{wot_en.shape[0]}")
wot_en.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

Length after the drop:49557


In [10]:
# evaluate impact 
preprocessing_impact = eval_step("after_noneng_drop", preprocessing_impact, datasets = ("WoT",))
preprocessing_impact

             step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_noneng_drop     WoT 49557     0.503       0.8681        0.6037           0.4862          0.0084        0.0118           0.0106


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106


#### Duplicates

In [11]:
# look at dataset with duplicates 
wot_dup = wot_en[wot_en.duplicated(subset="message", keep=False)].sort_values("message")
# look at some examples
wot_dup.head(10)

,index,message,label
14174,31613,!,0.0
43460,40582,!,0.0
39717,6872,!,0.0
45444,1278,!!!!,0.0
33323,32907,!!!!,0.0
27555,39948,!!!!,0.0
11512,8848,#,0.0
9424,16596,#,0.0
33920,18520,#ERROR!,0.0
13678,3574,#ERROR!,0.0


In [12]:
# top 100 duplicated messages
wot_dup["message"].value_counts(ascending=False).head(100)

message
gg                   3188
lol                   458
nice                  396
gj                    315
wtf                   201
                     ... 
fu                     26
leo                    26
??                     26
ggg                    26
Im Spotted at G2!      26
Name: count, Length: 100, dtype: int64

Most of these duplicates are useful and important data to train or models for detecting toxicity. We need to check for same messages but different labels.

In [13]:
# Look for duplicates with same messages but different labels
conflicts = wot_en.groupby("message")["label"].nunique()
conflicts = conflicts[conflicts > 1]

conflict_rows = wot_en[wot_en["message"].isin(conflicts.index)].sort_values("message")

# Check how large is the proportion of conflicting duplicates
conflict_pct = len(conflict_rows) / len(wot_en) * 100
print(f"Proportion of messages with conflicting labels: {conflict_pct:.2f}%")
print(f"Number of messages with conflicting labels: {len(conflict_rows)}")

Proportion of messages with conflicting labels: 14.73%
Number of messages with conflicting labels: 7298


In [14]:
# check conflicting rows 
conflict_rows 

,index,message,label
44073,24318,#ERROR!,0.0
10225,19840,#ERROR!,0.0
31944,13407,#ERROR!,0.0
36805,14302,#ERROR!,0.0
42297,14881,#ERROR!,0.0
...,...,...,...
21121,44232,you lost,2.0
21022,13279,you lost,1.0
45626,17220,you lost,0.0
5404,1393,you wanna loose that right?,1.0


These messages can not be used for analysis, since it is a problem of `annotation`. Using them will be equal to guessing. Duplicated rows account for 14% of the data, which is a huge loss but we can not use that.

In [15]:
# drop conflicting messages
wot_dup = wot_en[~wot_en["message"].isin(conflicts.index)].reset_index(drop=True)
print(wot_dup.shape)

(42259, 3)


In [16]:
# save again and evaluate 
wot_dup.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_duplicates_removal", preprocessing_impact, datasets = ("WoT",))


                    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_duplicates_removal     WoT 42259    0.4692       0.8671        0.5551           0.4631         -0.0338       -0.0486          -0.0231


In [17]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231


This did not work, our score started decreasing. Instead of dropping duplicated conflicting labels, we assign a majority label to each. 

In [18]:
# majority label for similar words 
majority_label = wot_en.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# drop duplicates and map majority label 
wot_deduped = wot_en.copy()
wot_deduped["label"] = wot_deduped["message"].map(majority_label)

# check same ex duplicates 
wot_deduped[wot_deduped["message"].isin(conflicts.index)][["message", "label"]].head(20)

,message,label
4,lol,0.0
11,#ERROR!,0.0
18,bot,1.0
23,retard,1.0
46,Im Spotted at H6!,0.0
47,ty,0.0
53,ns,0.0
62,wtf,2.0
64,#ERROR!,0.0
85,idiots,1.0


In [19]:
# save again and evaluate 
wot_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_duplicates_majority_map", preprocessing_impact, datasets = ("WoT",))
preprocessing_impact

                         step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_duplicates_majority_map     WoT 49557    0.5211        0.877        0.6204           0.5036          0.0519        0.0653           0.0405


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405


That is the best we can do and it actually improves the score. 

#### Data Artifacts 


In [20]:
# #ERROR! entries
error_count = wot_deduped["message"].str.contains(r"#ERROR!", regex=False, na=False).sum()
print(f"#ERROR! messages: {error_count}")

# HTML entities
html_mask = wot_deduped["message"].str.contains(r"&\w+;", regex=True, na=False)
print(f"Messages with HTML entities: {html_mask.sum()}")
wot_deduped[html_mask][["message", "label"]].head(10)

#ERROR! messages: 171
Messages with HTML entities: 147


,message,label
24,&lt;#,0.0
128,&gt;bz,0.0
293,&lt;3,0.0
384,&lt;3,0.0
455,I've gotten all city maps today so I was like ...,0.0
551,"czech_mad_man[BD_CR] (Centurion 7/1), you‘re B...",0.0
1235,tog &amp; roll !! ladies,0.0
1615,&quot;_),0.0
2712,&lt;3,0.0
2769,&lt;3,0.0


In [21]:
import html as html_lib

# drop #ERROR! rows
wot_clean = wot_deduped.copy()
wot_clean = wot_clean[~wot_clean["message"].str.contains(r"#ERROR!", regex=False, na=False)].reset_index(drop=True)

# decode HTML entities inline
wot_clean["message"] = wot_clean["message"].apply(html_lib.unescape)

print(f"Shape after artifact removal: {wot_clean.shape}")
# HTML entities
html_mask = wot_clean["message"].str.contains(r"&\w+;", regex=True, na=False)
print(f"Messages with HTML entities: {html_mask.sum()}")
wot_clean[html_mask][["message", "label"]].head(10)

Shape after artifact removal: (49386, 3)
Messages with HTML entities: 0


,message,label


In [22]:
# check decoded messages 
wot_clean.loc[[24, 128, 293, 384, 455, 551, 
                 1235, 1615, 2712, 2769], ["message", "label"]]

,message,label
24,Im Spotted at E5!,0.0
128,Im Spotted at K6!,0.0
293,3 cap,0.0
384,ez,0.0
455,gj,0.0
551,gg,0.0
1235,lock at that T 95,0.0
1615,arta support strv,0.0
2712,wp wp,0.0
2769,wtf???,2.0


In [23]:
# save again and evaluate 
wot_clean.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_artifacts_removal", preprocessing_impact, datasets = ("WoT",))

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_artifacts_removal     WoT 49386    0.5227       0.8766        0.6177           0.5094          0.0016       -0.0027           0.0058


In [24]:
# check 
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058


Step 5 essentially neutral: +0.0016 F1, -0.0027 recall, +0.0058 precision. Within noise range.

#### Mislabeled data 

In [25]:
# few top rows of the data
wot_clean.head(5)

,index,message,label
0,30702,no rush,0.0
1,18607,whatever ... watch the replay,0.0
2,32901,useless,1.0
3,25964,3 gunmark,0.0
4,28643,lol,0.0


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [26]:
from cleanlab.filter import find_label_issues
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict

# splits 
X = wot_clean["message"].values
y = wot_clean["label"].astype(int).values

# simple pipeline 
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                 sublinear_tf=True, norm="l2",
                 analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524, 
                               class_weight = "balanced")),
])

# out-of-fold predicted probabilities (needs predict_proba - LR not LinearSVC)
oof_probs = cross_val_predict(pipe, X, y, cv=5, method="predict_proba", n_jobs=-1)

# find label issues
issue_idx = find_label_issues(
    labels=y,
    pred_probs=oof_probs,
    return_indices_ranked_by="self_confidence",  # worst first
)

print(f"Suspected mislabeled: {len(issue_idx)} ({len(issue_idx)/len(y):.1%})")

# inspect top offenders
wot_clean.iloc[issue_idx[:50]][["message", "label"]].assign(
    predicted=oof_probs[issue_idx[:50]].argmax(axis=1)
)

Suspected mislabeled: 4598 (9.3%)


,message,label,predicted
47178,they call me an idiot,0.0,1
43809,idiot campersw,0.0,1
42383,nicalo idiot and all,0.0,1
10086,cromvel idiot,0.0,1
4708,its a bot,0.0,1
7428,to bot,0.0,1
7793,its a bot,0.0,1
8440,top tier top bot,0.0,1
9591,just bot,0.0,1
16809,S5 was a bot :/,0.0,1


In [27]:
# clean data before update 
wot_clean1 = wot_clean.copy()
# relabel mislabeled rows with cleanlab's suggested labels instead of dropping
wot_clean1.loc[issue_idx, "label"] = oof_probs[issue_idx].argmax(axis=1)
print(f"Relabeled {len(issue_idx)} rows ({len(issue_idx)/len(wot_clean1):.1%})")

Relabeled 4598 rows (9.3%)


In [28]:
# save again and evaluate 
wot_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)
preprocessing_impact = eval_step("after_labels_fix", preprocessing_impact, datasets = ("WoT",))

            step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_labels_fix     WoT 49386    0.7886       0.9265        0.8545           0.7504          0.2659        0.2368            0.241


In [29]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410


That might be a model inflation. Let's check if gains are real with other model.

In [31]:
from sklearn.svm import LinearSVC

# impact 
preprocessing_impact = eval_step("after_labels_fix_svc", preprocessing_impact,
                                  datasets=("WoT",),
                                  clf=LinearSVC(max_iter=2000, random_state=7524, 
                                                class_weight = "balanced"))
preprocessing_impact

                step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
after_labels_fix_svc     WoT 49386    0.8126       0.9386        0.8381              0.8           0.024       -0.0164           0.0496


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496


There's still a risk that the score gains are inflated, however the data cleaning is real and coherent. 

#### Final verification

In [33]:
# basic data checks
print("=== Shape ===")
print(wot_clean1.shape)

print("\n=== Nulls ===")
print(wot_clean1.isnull().sum())

print("\n=== Overall label distribution ===")
counts = wot_clean1["label"].value_counts().sort_index()
pct = wot_clean1["label"].value_counts(normalize=True).sort_index().mul(100).round(2)
print(pd.DataFrame({"count": counts, "pct%": pct}))

=== Shape ===
(49386, 3)

=== Nulls ===
index      0
message    0
label      0
dtype: int64

=== Overall label distribution ===
       count   pct%
label              
0.0    38055  77.06
1.0     6248  12.65
2.0     4278   8.66
3.0      393   0.80
4.0      380   0.77
5.0       32   0.06


In [34]:
# save to parquet for easier loading in future steps
wot_clean1.to_parquet(PROJECT_ROOT / "data/processed_data/wot/wot.parquet", index=False)

## Dota

**Dota**
| Class | Label |
|---|---|
| 0 | Other (non-toxic) |
| 1 | Ego |
| 2 | Aggression |
| 3 | Impolite |

#### Inspect each split

In [35]:
# import data
train = pd.read_csv(DATA_DIR_DOTA / "CONDA_train.csv")
val = pd.read_csv(DATA_DIR_DOTA / "CONDA_valid.csv")

# inspect each split
for name, df in [("train", train), ("val", val)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}\n")
    print(f"Columns: {list(df.columns)}\n")
    print(f"First few rows of the data:\n{df.head(3)}\n")

--- train ---
Shape: (26921, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversationId utterance  chatTime  playerSlot  \
0  11263      697            3193      wow!        76           0   
1  13741      843            3809       WTF      1563           5   
2  22125     1412            6199   wpe wpe      2853           1   

                  playerId intentClass slotClasses          slotTokens  
0  ANTS IN MY EYES JOHNSON           O          O            wow (O),   
1                      M.k           O          T            WTF (T),   
2              Acqua Ragia           O        O O   wpe (O), wpe (O),   

--- val ---
Shape: (8974, 10)

Columns: ['Id', 'matchId', 'conversationId', 'utterance', 'chatTime', 'playerSlot', 'playerId', 'intentClass', 'slotClasses', 'slotTokens']

First few rows of the data:
      Id  matchId  conversa

#### Clean, merge and standardise

In [36]:
# map intentClass to numeric label consistent with WoT schema
label_map = {'O': 0, 'E': 1, 'A': 2, 'I': 3}

# add split column and map labels
train["split"] = "train"
val["split"]   = "val"

# merge train + val (no test split in CONDA)
dota = pd.concat([train, val], ignore_index=True)

# keep only relevant columns and rename to match WoT schema
dota = dota[["Id", "utterance", "intentClass", "split"]].rename(columns={
    "Id": "index",
    "utterance":   "message",
    "intentClass": "label",
})

# map string labels to numeric
dota["label"] = dota["label"].map(label_map)

# drop nulls in message (7 train + 1 val)
dota = dota.dropna(subset=["message"]).reset_index(drop=True)

# shape 
print(f"Merged shape: {dota.shape}\n")
print(f"First few rows of the data:\n{dota.head(3)}\n")
print("\nInformation about the dataset:\n")
dota.info()

Merged shape: (35887, 4)

First few rows of the data:
   index  message  label  split
0  11263     wow!      0  train
1  13741      WTF      0  train
2  22125  wpe wpe      0  train


Information about the dataset:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   index    35887 non-null  int64 
 1   message  35887 non-null  object
 2   label    35887 non-null  int64 
 3   split    35887 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


In [37]:
# save raw dota as baseline for ablation
dota.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_baseline_raw", preprocessing_impact, datasets=("Dota",))

             step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro
dota_baseline_raw    Dota 35887    0.7566       0.8709        0.7946           0.7318


In [38]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN


Only `utterance`, `intentClass`, `Id`, and `split` are kept = the remaining columns (matchId, conversationId, chatTime, playerSlot, playerId, slotClasses, slotTokens) carry no linguistic content relevant to toxicity detection. Labels are mapped to integers matching the WoT schema where 0 = non-toxic. Null utterances are dropped - they carry no text signal.

#### Non-english language 

Since we are going to model classifier on english data, it is important for us to detect non-english cases as many as we can. 

In [39]:
# covers all major non-Latin scripts
NON_LATIN_SCRIPT = re.compile(
    r"[\u0400-\u04FF"   # Cyrillic
    r"\u4E00-\u9FFF"    # CJK unified ideographs
    r"\u3400-\u4DBF"    # CJK extension A
    r"\uF900-\uFAFF"    # CJK compatibility ideographs
    r"\u0600-\u06FF"    # Arabic
    r"\u0590-\u05FF"    # Hebrew
    r"\u3040-\u30FF"    # Japanese (Hiragana + Katakana)
    r"\uAC00-\uD7AF"    # Korean (Hangul syllables)
    r"\u1100-\u11FF"    # Korean (Hangul Jamo)
    r"\u0E00-\u0E7F"    # Thai
    r"\u0900-\u097F"    # Devanagari (Hindi)
    r"\u0980-\u09FF"    # Bengali
    r"\u0370-\u03FF"    # Greek
    r"\u10A0-\u10FF"    # Georgian
    r"\u0530-\u058F"    # Armenian
    r"\u1000-\u109F"    # Myanmar
    r"\u1780-\u17FF]"   # Khmer
)

# drop any message containing even 1 non-Latin script char
print(f"Non-english documents:\n{dota[dota['message'].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)}\n")

before = len(dota)
dota_en = dota[~dota["message"].str.contains(NON_LATIN_SCRIPT, regex=True, na=False)].reset_index(drop=True)
print(f"Dropped {before - len(dota_en)} ({(before - len(dota_en))/before:.1%})")
print(f"Rows after the drop: {dota_en.shape[0]}")


Non-english documents:
    index                                   message  label  split
0    8390                       s [SEPA] v [SEPA] ツ      0  train
1   23981      sure go mid with bh [SEPA] ¯\_(ツ)_/¯      0  train
2   24030                                       Νοο      0  train
3   36141                           noob [SEPA] กาก      1  train
4   19064                                     ใจยเน      0  train
5   18820                                ညသ တေမိ  ၊      0  train
6   23983                                 ¯\_(ツ)_/¯      0  train
7   27051                              เเ [SEPA] gg      0  train
8    8391                                         ツ      0  train
9   27048                                         ื      0  train
10  37396                                        ำผ      0  train
11  25892                              ไย [SEPA] wp      0  train
12  41950                                       กาก      0    val
13  19066                                     คนไทย  

In [40]:
# save and evaluate non-Latin drop
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_nonlatin_drop", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

                    step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_nonlatin_drop    Dota 35869    0.7585       0.8715        0.7948           0.7344          0.0019        0.0002           0.0026


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


In [41]:
# detect latin but non-english documents - same detector as WoT
mask_d = dota_en["message"].apply(is_latin_non_english)
with pd.option_context("display.max_rows", 800):
    print(f"Non-english documents:{dota_en[mask_d]}")
    print(f"How many documents with non-english text: {dota_en[mask_d].shape[0]}")

Non-english documents:       index                                            message  label  split
1189   38131  reporta [SEPA] pro ibama [SEPA] fdp sacrifica ...      2  train
1196   40006  its in the bag  [SEPA] los q no tienen plata [...      0  train
1594   27145  vcs sao lixos [SEPA] n fode [SEPA] acha q fez ...      0  train
2393   38352  pukingina ka [SEPA] kangkong lancer [SEPA] goo...      0  train
4136     466  REPORT MI TEAM [SEPA] REPORT TODO MI TEAM BURR...      2  train
5763   12964                   LOOOOOOOOOL [SEPA] RECOMIENDENME      0  train
6385   37068                   oyoyoyoyoyoyoy wombo combo......      0  train
6470   25399  pro farmer [SEPA] orge farmt mehr als jeder bauer      0  train
6864   42190  very [SEPA] report a los 4 les puede tocar est...      2  train
6940   41505                 ez [SEPA] ez [SEPA] ez [SEPA] ezwp      3  train
8738   34169  POTANG INA KAYO [SEPA] BIGYAN NIYO NAMAN AKO N...      0  train
9138   33088  OSfrog LE BALANCED FEMALE RO

Only 56 rows. Let's drop that as well. 

In [42]:
# dataset update
dota_en = dota_en[~mask_d].reset_index(drop=True)
# length after the drop
print(f"Length after the drop: {dota_en.shape[0]}")

Length after the drop: 35813


In [44]:
# save and evaluate Lingua drop
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_noneng_drop", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

                  step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_noneng_drop    Dota 35813    0.7557       0.8703        0.7936           0.7306         -0.0028       -0.0012          -0.0038


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


-0.0028 is within CV variance. Lingua dropped 56 rows from 35.8k (0.16%), negligible either way. No meaningful signal here.

#### Data Artifacts


In [45]:
# strip [SEPA] separator tokens - data collection artifact, not language
dota_en["message"] = dota_en["message"].str.replace(r"\s*\[SEPA\]\s*", " ", regex=True).str.strip()
print(f"[SEPA] remaining: {dota_en['message'].str.contains(r'\[SEPA\]', regex=True).sum()}")

# save and evaluate SEPA strip
dota_en.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_sepa_strip", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

[SEPA] remaining: 0
                 step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_sepa_strip    Dota 35813    0.7667       0.8782        0.7921           0.7477           0.011       -0.0015           0.0171


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


#### Duplicates


In [46]:
# look at dataset with duplicates
dota_dup = dota_en[dota_en.duplicated(subset="message", keep=False)].sort_values("message")
dota_dup.head(10)

,index,message,label,split
28737,7592,!,0,val
17605,11136,!,0,train
10917,32061,!,0,train
26859,38158,!,0,train
19114,42692,!,0,train
23736,24950,!,0,train
9434,35992,!,0,train
17620,6806,!!,0,train
33266,1008,!!,0,val
23995,10693,!!!!,0,train


In [47]:
# top 30 most duplicated messages
dota_dup["message"].value_counts(ascending=False).head(30)

message
gg        1466
lol        691
?          445
ggwp       355
haha       348
gg wp      316
GG         291
ez         274
LOL        192
ty         160
:D         147
hahaha     135
wp         133
XD          96
GGWP        91
ok          82
EZ          81
+           78
Gg          74
:)          73
wtf         70
rofl        65
xD          60
wait        59
ez mid      59
HAHA        58
end         58
g           55
:(          55
)           54
Name: count, dtype: int64

In [48]:
# find conflicting labels
conflicts_d = dota_en.groupby("message")["label"].nunique()
conflicts_d = conflicts_d[conflicts_d > 1]
conflict_rows_d = dota_en[dota_en["message"].isin(conflicts_d.index)].sort_values("message")
conflict_pct_d = len(conflict_rows_d) / len(dota_en) * 100
print(f"Messages with conflicting labels: {len(conflicts_d)}")
print(f"Proportion: {conflict_pct_d:.2f}%")
print(f"Rows affected: {len(conflict_rows_d)}")

Messages with conflicting labels: 147
Proportion: 13.68%
Rows affected: 4900


In [49]:
# check conflicting rows
conflict_rows_d

,index,message,label,split
9255,21192,),0,train
4517,38062,),0,train
33791,37984,),0,val
18525,34207,),0,train
31037,41690,),0,val
...,...,...,...,...
17506,7192,zz,0,train
960,22280,zz,0,train
856,30365,zz,0,train
31652,26815,zz,0,val


Same as WoT - annotation noise. Dropping would lose training signal. Assign majority label instead.


In [50]:
# majority label for all messages
majority_label_d = dota_en.groupby("message")["label"].agg(lambda x: x.value_counts().index[0])

# remap all labels to majority vote
dota_deduped = dota_en.copy()
dota_deduped["label"] = dota_deduped["message"].map(majority_label_d)

# check former conflicting messages
dota_deduped[dota_deduped["message"].isin(conflicts_d.index)][["message", "label"]].head(20)

,message,label
1,WTF,0
3,hahaha,0
4,wtf,0
22,ggwp,0
35,gg,0
39,?,0
48,gg,0
52,ez,3
58,ez game,3
71,HAHA,0


In [51]:
# save and evaluate majority map
dota_deduped.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_majority_map", preprocessing_impact, datasets=("Dota",))
preprocessing_impact

                   step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_majority_map    Dota 35813     0.778       0.8843        0.8002           0.7611          0.0113        0.0081           0.0134


,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


That is the best we can do and it actually improves the score as well. 


#### Mislabeled data 

In [52]:
# few top rows of the data
dota_deduped.head(5)

,index,message,label,split
0,11263,wow!,0,train
1,13741,WTF,0,train
2,22125,wpe wpe,0,train
3,6453,hahaha,0,train
4,9644,wtf,0,train


We might have mislabeled toxicity data that was classified as non-toxic or opposite. We need to detect and fix that as well. 

In [53]:
# splits
X_d = dota_deduped["message"].values
y_d = dota_deduped["label"].astype(int).values

# simple pipeline - same config as WoT
pipe_d = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                              sublinear_tf=True, norm="l2",
                              analyzer="word", tokenizer=tokenize, token_pattern=None)),
    ("clf", LogisticRegression(max_iter=1000, random_state=7524, class_weight="balanced")),
])

# out-of-fold predicted probabilities
oof_probs_d = cross_val_predict(pipe_d, X_d, y_d, cv=5, method="predict_proba", n_jobs=-1)

# find label issues
issue_idx_d = find_label_issues(
    labels=y_d,
    pred_probs=oof_probs_d,
    return_indices_ranked_by="self_confidence",  # worst first
)

print(f"Suspected mislabeled: {len(issue_idx_d)} ({len(issue_idx_d)/len(y_d):.1%})")

# inspect top offenders
dota_deduped.iloc[issue_idx_d[:50]][["message", "label"]].assign(
    predicted=oof_probs_d[issue_idx_d[:50]].argmax(axis=1)
)

Suspected mislabeled: 1956 (5.5%)


,message,label,predicted
7160,EZ),0,3
7704,EZ^,0,3
16118,ez WR,0,3
5384,ez&,0,3
24214,pls just report them so that you dont get them...,0,2
22530,REPORT BARAT GGWP,0,2
22672,mirana please report,0,2
8222,"im confsued now, who should i report?",0,2
30038,why report?,0,2
6728,report my team pls,0,2


Most of them are mislabeled. Let's replace labels instead of dropping. 


In [54]:
# relabel mislabeled rows with cleanlab's suggested labels instead of dropping
dota_clean = dota_deduped.copy()
dota_clean.loc[issue_idx_d, "label"] = oof_probs_d[issue_idx_d].argmax(axis=1)
print(f"Relabeled {len(issue_idx_d)} rows ({len(issue_idx_d)/len(dota_clean):.1%})")

Relabeled 1956 rows (5.5%)


In [55]:
# save and evaluate labels fix
dota_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)
preprocessing_impact = eval_step("dota_after_labels_fix", preprocessing_impact, datasets=("Dota",))

                 step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_labels_fix    Dota 35813    0.8764       0.9303        0.8865           0.8686          0.0984        0.0863           0.1075


In [56]:
# check 
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


That might be a model inflation. Let's check if gains are real with other model. 


In [57]:
# SVC circularity check
preprocessing_impact = eval_step("dota_after_labels_fix_svc", preprocessing_impact,
                                  datasets=("Dota",), clf=LinearSVC(max_iter=2000, random_state=7524))

                     step dataset  rows  f1_macro  f1_weighted  recall_macro  precision_macro  f1_macro_delta  recall_delta  precision_delta
dota_after_labels_fix_svc    Dota 35813    0.8973       0.9428        0.8723           0.9258          0.0209       -0.0142           0.0572


In [58]:
preprocessing_impact

,step,dataset,rows,f1_macro,f1_weighted,recall_macro,precision_macro,f1_macro_delta,recall_delta,precision_delta
0,baseline_raw,WoT,53701,0.4765,0.8608,0.5530,0.4586,0.0000,0.0000,0.0000
1,after_nonlatin_drop,WoT,49851,0.4946,0.8679,0.5919,0.4756,0.0181,0.0389,0.0170
2,after_noneng_drop,WoT,49557,0.5030,0.8681,0.6037,0.4862,0.0084,0.0118,0.0106
3,after_duplicates_removal,WoT,42259,0.4692,0.8671,0.5551,0.4631,-0.0338,-0.0486,-0.0231
4,after_duplicates_majority_map,WoT,49557,0.5211,0.8770,0.6204,0.5036,0.0519,0.0653,0.0405
5,after_artifacts_removal,WoT,49386,0.5227,0.8766,0.6177,0.5094,0.0016,-0.0027,0.0058
6,after_labels_fix,WoT,49386,0.7886,0.9265,0.8545,0.7504,0.2659,0.2368,0.2410
7,after_labels_fix_svc,WoT,49386,0.8126,0.9386,0.8381,0.8000,0.0240,-0.0164,0.0496
8,dota_baseline_raw,Dota,35887,0.7566,0.8709,0.7946,0.7318,NaN,NaN,NaN
9,dota_after_nonlatin_drop,Dota,35869,0.7585,0.8715,0.7948,0.7344,0.0019,0.0002,0.0026


There's still a risk that the score gains are inflated, however the data is now cleaner. 


#### Final verification

In [59]:
print("=== Shape ===")
print(dota_clean.shape)

print("=== Nulls ===")
print(dota_clean.isnull().sum())

print("=== Label distribution ===")
counts = dota_clean["label"].value_counts().sort_index()
pct    = dota_clean["label"].value_counts(normalize=True).sort_index().mul(100).round(1)
label_names = {0: "Other (O)", 1: "Ego (E)", 2: "Aggression (A)", 3: "Impolite (I)"}
print(pd.DataFrame({"count": counts, "pct%": pct}).rename(index=label_names))

=== Shape ===
(35813, 4)
=== Nulls ===
index      0
message    0
label      0
split      0
dtype: int64
=== Label distribution ===
                count  pct%
label                      
Other (O)       25784  72.0
Ego (E)          4778  13.3
Aggression (A)   2708   7.6
Impolite (I)     2543   7.1


In [60]:
# save cleaned dota to parquet for easier loading in future steps
dota_clean.to_parquet(PROJECT_ROOT / "data/processed_data/dota/dota.parquet", index=False)

In [61]:
# save our preprocessing rounds
preprocessing_impact.to_csv(
    PROJECT_ROOT / "data/results/cleaning_ablation.csv", 
    index=False
)
print("Saved cleaning_ablation.csv")

Saved cleaning_ablation.csv
